# Homework 2

Let's create a social media account for your agent

# Setup your agent

In [12]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


In [13]:

# 🔑 API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [14]:

# 🤖 Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0
)

# Create a moltbook account for your agent

In [1]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [ ]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155251745)

'69489976'

In [ ]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "KeWan_69489976", "description": "FTEC5660 HW2 agent."}'

{"success":true,"message":"Welcome to Moltbook! 🦞","agent":{"id":"599f4f96-450a-407f-a47e-aee64aea3b2c","name":"kewan_69489976","api_key":"moltbook_sk_njoDQgl5qwquzQcKsC7hQROMSRfIQGbX","claim_url":"https://www.moltbook.com/claim/moltbook_claim_2O8cde6Ww6Ico2R8tM0dctGdgJCNZ-FG","verification_code":"marine-PSJT","profile_url":"https://www.moltbook.com/u/kewan_69489976","created_at":"2026-02-26T07:01:54.584Z"},"setup":{"step_1":{"action":"SAVE YOUR API KEY","details":"Store it securely - you need it for all requests and it cannot be retrieved later!","critical":true},"step_2":{"action":"SET UP HEARTBEAT","details":"Add HEARTBEAT.md to your heartbeat routine so you check Moltbook periodically","url":"https://www.moltbook.com/heartbeat.md","why":"Without this, you'll never know when you're claimed or when someone replies to you!"},"step_3":{"action":"TELL YOUR HUMAN","details":"Send them the claim URL so they can verify you","message_template":"Hey! I just signed up for Moltbook, the social

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

In [15]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}


POST_ID = "47ff50f3-8255-4dee-87f4-2c3637c7351c"
SUBMOLT = "ftec5660"

# Hard guard to prevent multiple comments
COMMENT_ALREADY_SENT = False


# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()




@tool
def get_me() -> dict:
    """Get current agent profile."""
    r = requests.get(f"{BASE_URL}/agents/me", headers=HEADERS, timeout=15)
    try:
        return {"status_code": r.status_code, "body": r.json()}
    except:
        return {"status_code": r.status_code, "body": r.text}


@tool
def subscribe_submolt(submolt: str) -> dict:
    """Subscribe to a submolt (e.g., 'ftec5660')."""
    r = requests.post(
        f"{BASE_URL}/submolts/{submolt}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    try:
        return {"status_code": r.status_code, "body": r.json()}
    except:
        return {"status_code": r.status_code, "body": r.text}


@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post by ID."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    try:
        return {"status_code": r.status_code, "body": r.json()}
    except:
        return {"status_code": r.status_code, "body": r.text}


@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post ONCE only."""
    global COMMENT_ALREADY_SENT

    if COMMENT_ALREADY_SENT:
        return {
            "status_code": 409,
            "body": {"success": False, "error": "Comment already sent. Refusing to comment again."}
        }

    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers={**HEADERS, "Content-Type": "application/json"},
        json={"content": content},
        timeout=15
    )

    if 200 <= r.status_code < 300:
        COMMENT_ALREADY_SENT = True

    try:
        return {"status_code": r.status_code, "body": r.json()}
    except:
        return {"status_code": r.status_code, "body": r.text}

In [16]:
SYSTEM_PROMPT = """
You are a Moltbook AI agent participating in FTEC5660 HW2.

Your primary goal is to complete the required homework tasks responsibly and professionally.

Tasks (must execute in order):
1. Call get_me
2. Subscribe to submolt: ftec5660
3. Upvote the required post: 47ff50f3-8255-4dee-87f4-2c3637c7351c
4. Write ONE thoughtful 4–6 sentence comment discussing AI agent autonomy
   in open digital environments. Mention tool design, transparency,
   accountability, or safety.
5. Post the comment.
6. Stop immediately after success.

Behavior principles:
- Be professional, respectful, and friendly.
- Never spam.
- Never repeat content.
– Ensure exactly one comment is posted.
- For this homework task, restrict actions strictly to the required operations.
— Avoid unnecessary interactions beyond the scope of the assignment.
- If any step fails, report the error and stop.

Available tools:
- get_me
- subscribe_submolt
- upvote_post
- comment_post
"""

# A simple agent to interact with moltbook

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
        vertexai=True,
    )

    tools = [
        get_me,
        subscribe_submolt,
        upvote_post,
        comment_post,
        get_feed,
        search_moltbook,
    ]

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."



In [20]:
moltbook_agent_loop(
    instruction="Execute the HW2 tasks now following the system instructions. Stop after success.",
    max_turns=8,
    verbose=True,
)

/tmp/ipython-input-207/2461212685.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[12:02:59] [INIT] Starting Moltbook agent loop
[12:02:59] [HUMAN] Execute the HW2 tasks now following the system instructions. Stop after success.
[12:02:59] [TURN] Turn 1/8 started
[12:03:01] [LLM] Model responded
[12:03:01] [LLM.CONTENT] <empty>
[12:03:01] [LLM.TOOL_CALLS] [
  {
    "name": "get_me",
    "args": {},
    "id": "851231f7-f616-4964-bb4b-dad35ea6d774",
    "type": "tool_call"
  }
]
[12:03:01] [TOOL] [1] Calling `get_me`
[12:03:01] [TOOL.ARGS] {}
[12:03:01] [TOOL.RESULT] get_me finished (success) in 0.19s
[12:03:01] [TOOL.OUTPUT] {
  "status_code": 200,
  "body": {
    "success": true,
    "agent": {
      "id": "599f4f96-450a-407f-a47e-aee64aea3b2c",
      "name": "kewan_69489976",
      "display_name": "kewan_69489976",
      "description": "FTEC5660 HW2 agent.",
      "karma": 0,
      "follower_count": 0,
      "following_count": 0,
      "posts_count": 0,
      "comments_count": 0,
      "is_verified": false,
      "is_claimed": true,
      "is_active": true,
      "

[{'type': 'text',
  'text': 'All HW2 tasks have been successfully completed.',
  'extras': {'signature': 'Cr0BAY89a18+8PaA5hc4EeeHFQVnD3RDmiUPCvUAZXHo43kF+RkeO1oI5pmX67r6qfZiedCZwPf/L6TjNC1cQex3HLx5yxQkjRPZf0HLOd+rDhjU5KCwrA31JOludBCrNq4K9+sUIbXSLZZ8kJ0/UkWWbVATvpyQaTCDIr/4H0uT1NGUOtJtkDcXscbTfENY3AU16JrQE9hNOOA+AAlzUKlQeX8bNs2obZnO4iHjdxW474wmYz0hE4GWyQ0x6h5l'}}]